# Notebook 02 — Preprocessing

Maintenant qu'on a exploré les données dans le notebook 01, on passe au nettoyage. Le but c'est de transformer le dataset brut en un dataset propre prêt pour la modélisation. J'utilise un pipeline avec ColumnTransformer comme on a vu dans le TP-Arbre (2ème solution).

## Section 0 — Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")

## Section 1 — Chargement des données brutes

In [2]:
df_brut = pd.read_csv("../data/raw/instagram_brut.csv")
print(df_brut.shape)
print(df_brut.dtypes)

(797, 13)
edge_followed_by        float64
edge_follow             float64
username_length         float64
username_has_number       int64
full_name_has_number      int64
full_name_length        float64
is_private                int64
is_joined_recently        int64
has_channel               int64
is_business_account       int64
has_guides                int64
has_external_url          int64
is_fake                   int64
dtype: object


On retrouve bien le dataset brut avec ses 797 lignes et 13 colonnes. Comme vu dans le notebook 01, edge_followed_by est en object alors qu'il devrait être en float — c'est à cause des "N/A" qu'on avait injectés. On va corriger ça avant tout.

## Section 2 — Conversion du type de edge_followed_by

In [3]:
df_brut["edge_followed_by"] = pd.to_numeric(df_brut["edge_followed_by"], errors="coerce")
print(df_brut.dtypes)

edge_followed_by        float64
edge_follow             float64
username_length         float64
username_has_number       int64
full_name_has_number      int64
full_name_length        float64
is_private                int64
is_joined_recently        int64
has_channel               int64
is_business_account       int64
has_guides                int64
has_external_url          int64
is_fake                   int64
dtype: object


J'ai utilisé pd.to_numeric avec errors="coerce" comme dans l'exercice ex1.py du cours. Ça convertit les valeurs en float, et toutes les valeurs non convertibles (comme "N/A") deviennent automatiquement des NaN. Maintenant la colonne est bien en float64.

## Section 3 — Séparation features / cible

In [4]:
X = df_brut.drop("is_fake", axis=1)
y = df_brut["is_fake"]
print(X.shape)
print(y.shape)

(797, 12)
(797,)


Comme dans le TP-Arbre, je sépare les features (X) de la variable cible (y). Ici, is_fake est notre cible.

## Section 4 — Identification des colonnes numériques et catégorielles

In [5]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
print(numeric_features)

['edge_followed_by', 'edge_follow', 'username_length', 'username_has_number', 'full_name_has_number', 'full_name_length', 'is_private', 'is_joined_recently', 'has_channel', 'is_business_account', 'has_guides', 'has_external_url']


In [6]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
print(categorical_features)

[]


J'utilise select_dtypes comme dans le TP-Arbre pour distinguer automatiquement les colonnes numériques et catégorielles. Toutes mes colonnes sont numériques (int64 ou float64), donc la liste des catégorielles est vide. Je garde quand même le code pour les catégorielles : si on rajoutait une feature texte plus tard, le pipeline marcherait toujours.

## Section 5 — Création des transformers

In [7]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

In [8]:
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder())
])

J'ai créé deux pipelines comme dans le TP-Arbre. Le numeric_transformer fait deux choses : il remplace les NaN par la moyenne (SimpleImputer avec strategy="mean") puis il normalise avec StandardScaler. Le categorical_transformer applique juste un OneHotEncoder. La force du Pipeline c'est que ça enchaîne les opérations dans le bon ordre.

## Section 6 — Création du ColumnTransformer

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

Le ColumnTransformer applique chaque transformer sur le bon sous-ensemble de colonnes : numeric_transformer sur les colonnes numériques, categorical_transformer sur les colonnes catégorielles. C'est exactement le pattern du TP-Arbre.

## Section 7 — Application du preprocessor

In [10]:
X_propre = preprocessor.fit_transform(X)
print(X_propre.shape)

(797, 12)


J'applique le preprocessor à X avec fit_transform. Ça applique toutes les étapes (imputation + standardisation) en une seule fois. Le résultat X_propre est un tableau numpy avec les mêmes dimensions que X.

## Section 8 — Reconstruction du DataFrame propre

In [11]:
df_propre = pd.DataFrame(X_propre, columns=numeric_features)
df_propre["is_fake"] = y.values
print(df_propre.shape)
print(df_propre.head())

(797, 13)
   edge_followed_by  edge_follow  username_length  username_has_number  \
0         -0.034318    -0.495721        -0.060895             0.742013   
1         -0.062244     1.928627        -0.125656             0.742013   
2         -0.062244    -0.509554        -0.077085            -1.347685   
3         -0.062244     1.994336        -0.109466             0.742013   
4         -0.062244    -0.274382        -0.093275            -1.347685   

   full_name_has_number  full_name_length  is_private  is_joined_recently  \
0              2.838453          0.120402   -0.475556           -0.750163   
1             -0.352305         -0.227937   -0.475556            1.333043   
2             -0.352305         -0.227937   -0.475556           -0.750163   
3             -0.352305         -0.227937   -0.475556           -0.750163   
4             -0.352305          0.066812    2.102800           -0.750163   

   has_channel  is_business_account  has_guides  has_external_url  is_fake  
0    

Je reconstruis un DataFrame avec les colonnes numériques transformées, puis je rajoute la colonne is_fake (la cible). Comme on n'a pas de colonnes catégorielles, les noms de colonnes restent les mêmes.

In [12]:
print(df_propre.isnull().sum())

edge_followed_by        0
edge_follow             0
username_length         0
username_has_number     0
full_name_has_number    0
full_name_length        0
is_private              0
is_joined_recently      0
has_channel             0
is_business_account     0
has_guides              0
has_external_url        0
is_fake                 0
dtype: int64


Je vérifie qu'il n'y a plus aucun NaN — l'imputer a bien fait son travail.

## Section 9 — Sauvegarde du dataset propre

In [13]:
df_propre.to_csv("../data/processed/instagram_propre.csv", index=False)
print("Dataset propre sauvegardé.")

Dataset propre sauvegardé.


Le fichier instagram_propre.csv est maintenant disponible dans data/processed/. C'est ce dataset qui sera utilisé dans le notebook 03 pour la modélisation.

## Section 10 — Conclusion

Voilà ce qu'on a fait dans ce notebook.

**Étapes appliquées** :
- Conversion du type de edge_followed_by (object vers float) avec pd.to_numeric
- Imputation des NaN avec SimpleImputer (strategy="mean")
- Normalisation avec StandardScaler

**Outils utilisés** : Pipeline et ColumnTransformer, comme dans la 2ème solution du TP-Arbre. C'est un pattern propre et réutilisable — on pourra brancher directement ce preprocessor sur les modèles dans le notebook 03.

**Note sur les valeurs aberrantes** : on les avait repérées dans le notebook 01 (username_length jusqu'à 999, full_name_length jusqu'à 700). Le StandardScaler va atténuer leur impact mais pas les supprimer. Si les modèles ont du mal, on reviendra ici pour les traiter spécifiquement.

**Note sur les doublons** : on en avait 17 dans le dataset brut. On les laisse pour l'instant, ça ne devrait pas trop impacter les résultats vu la taille du dataset.

**Sortie** : data/processed/instagram_propre.csv, prêt pour la modélisation.